# AOU-0 — Pre-fire validation pre-check. Phase M3 / cross-Wave.

Always run BEFORE any AOU-1 (or AOU-1-chr22-smoke) or AOU-2 fire. Compute-free (gsutil + bash subprocess only; no Hail/Spark). Expected runtime ~2 min; expected cost ~$0.05 (paused-cluster + minimal gsutil).

**Purpose:**
1. Confirm AoU clone HEAD includes the m3-W1 Track 4 defensive-code patches (commits `59e914b..bfe5f0e`) — if absent, halt before any compute fires.
2. Confirm AoU env vars are set (`WORKSPACE_BUCKET`, `GOOGLE_PROJECT`, `WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH`).
3. Inventory the catastrophe MTs in the workspace bucket (`mt_afr_qc.mt`, `mt_afr_pca_selfid_qc.mt`, `mt_eur_qc.mt`).
4. Resolve the kill-as-culprit vs Hail-finalize-on-empty-contents hypothesis via `_SUCCESS` mtime parse (per `[[feedback_w1_catastrophe_hypothesis_distinguisher]]`).
5. Confirm the entries/entries/parts/ pattern (catastrophe still present? or has bucket been purged?).
6. Emit an action-routing summary based on findings.
7. **CHECK C** — env-derive the AUX base from the WGS path and confirm `ancestry_preds.tsv` + `relatedness_flagged_samples.tsv` list on the current CDR (closes the manual CHECK-C gate; the driver auto-derives per DEC-2026-06-01).

**Run on Researcher Workbench 2.0 OR Legacy** — script is platform-agnostic. Cell 1 verifies which platform is in use.

**Cross-references:**
- `.planning/quick/260528-jvd-land-m3-w1-track-4-defensive-code-patche/260528-jvd-SUMMARY.md` — Track 4 patches
- `.planning/phases/m3-aou-afr-ld-panel-build/m3-CONTEXT.md` D-M3-10 — verification protocol
- `.planning/debug/m3-W1-empty-mt-catastrophe.md` — root cause analysis
- `[[feedback_aou_success_marker_not_evidence_of_data]]`
- `[[feedback_hail_checkpoint_contract_violation]]`
- `[[feedback_w1_catastrophe_hypothesis_distinguisher]]`


In [ ]:
# Cell 1 — AoU clone state + Track 4 patch presence + RW platform identification
import os
import subprocess
import sys

CLONE = '/home/jupyter/coloc_analysis'
assert os.path.isdir(CLONE), (
    f'AoU clone not found at {CLONE}; expected git clone of '
    f'https://github.com/carter-clinton/coloc_analysis (branch m3-W2-aou-deltas)'
)

def sh(cmd, cwd=CLONE):
    return subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)

head = sh('git rev-parse HEAD').stdout.strip()
branch = sh('git rev-parse --abbrev-ref HEAD').stdout.strip()
remote_head = sh('git rev-parse origin/m3-W2-aou-deltas').stdout.strip()
behind = sh('git rev-list --count HEAD..origin/m3-W2-aou-deltas').stdout.strip()
ahead = sh('git rev-list --count origin/m3-W2-aou-deltas..HEAD').stdout.strip()

print('=== AoU clone state ===')
print(f'  HEAD                          : {head}')
print(f'  Branch                        : {branch}')
print(f'  origin/m3-W2-aou-deltas HEAD  : {remote_head}')
print(f'  Behind                        : {behind}')
print(f'  Ahead                         : {ahead}')

if int(behind) > 0:
    print()
    print('!! WARN: AoU clone is behind origin. Run:')
    print('     cd /home/jupyter/coloc_analysis && git pull origin m3-W2-aou-deltas')
    print('   then re-fire this notebook from Cell 1.')

print()
print('=== Track 4 patch presence (must all be > 0) ===')
src = f'{CLONE}/src/python/aou_ld_panel.py'
with open(src) as f:
    content = f.read()
checks = {
    '_validate_checkpoint_populated': content.count('_validate_checkpoint_populated'),
    '_assert_checkpoint_nonempty': content.count('_assert_checkpoint_nonempty'),
    '_resolve_aux_base': content.count('_resolve_aux_base'),
    'MIN_ENTRIES_FILE_BYTES': content.count('MIN_ENTRIES_FILE_BYTES'),
    "_has_checkpoint (legacy)": content.count('_has_checkpoint'),
}
for k, v in checks.items():
    print(f'  grep -c {k!r} = {v}')
    assert v > 0, f'Track 4 helper {k} missing — clone is stale or wrong branch'
print()
print('  Track 4 patches present. Proceeding.')
print()
print('=== RW platform identification ===')
# RW 2.0 vs Legacy: best signal is the env var TERRA_WORKSPACE_NAMESPACE
# (Legacy) vs WORKBENCH_VERSION (RW 2.0) — confirm against AoU docs at fire time.
# For now print the candidate identifiers:
for env_var in ['TERRA_WORKSPACE_NAMESPACE', 'WORKBENCH_VERSION', 'GOOGLE_CLOUD_NOTEBOOK_RUNTIME',
                'JUPYTER_SERVER_ROOT', 'OWNER_EMAIL']:
    print(f'  {env_var:32s} = {os.environ.get(env_var, "<unset>")}')



In [ ]:
# Cell 2 — AoU env-var assertions (required by all downstream notebooks)
REQUIRED = [
    'WORKSPACE_BUCKET',
    'GOOGLE_PROJECT',
    'WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH',
]
missing = [v for v in REQUIRED if v not in os.environ]
assert not missing, f'Missing required AoU env vars: {missing}'
print('=== AoU env vars ===')
for v in REQUIRED:
    print(f'  {v:36s} = {os.environ[v]}')

# Strip gs:// prefix per src/python/aou_ld_panel.py _normalize_bucket convention
BUCKET = os.environ['WORKSPACE_BUCKET'].removeprefix('gs://').strip('/')
print()
print(f'  BUCKET (normalized for gsutil) = {BUCKET}')

# CDR version inference from WGS_ACAF path
wgs = os.environ['WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH']
if '/v8/' in wgs:
    cdr = 'v8'
elif '/v9/' in wgs:
    cdr = 'v9'
else:
    cdr = '<unknown>'
print(f'  CDR version (inferred)         = {cdr}')

# --- CHECK C: env-derive the AUX base the SAME way the driver does, then
# --- confirm the two load-bearing aux tables actually LIST on this CDR.
# The driver auto-derives AUX_BASE from the WGS MT path (DEC-2026-06-01), so a
# CDR advance (v8->v9) needs NO code edit. This cell CONFIRMS the derived paths
# exist before any compute spend -- that is the entirety of "CHECK C" now.
sys.path.insert(0, f'{CLONE}/src/python')
from aou_ld_panel import _resolve_aux_base, AUX_BASE  # noqa: E402

aux_base = _resolve_aux_base(wgs)
anc = f'{aux_base}/ancestry/ancestry_preds.tsv'
rel = f'{aux_base}/relatedness/relatedness_flagged_samples.tsv'
print()
print('=== CHECK C: AUX base resolution (env-derived from WGS path) ===')
print(f'  resolved AUX_BASE   = {aux_base}')
print(f'  ancestry_preds      = {anc}')
print(f'  relatedness_flagged = {rel}')
print(f'  env-derived         = {aux_base != AUX_BASE}  (hardcoded fallback = {AUX_BASE})')
print()
print('=== CHECK C: confirm aux tables LIST on this CDR (gsutil -u, requester-pays) ===')
proj = os.environ['GOOGLE_PROJECT']
check_c_ok = True
for label, path in [('ancestry_preds', anc), ('relatedness_flagged', rel)]:
    r = sh(f'gsutil -u {proj} ls -l "{path}"')
    found = (r.returncode == 0) and (path.rsplit('/', 1)[-1] in r.stdout)
    check_c_ok = check_c_ok and found
    print(f"  [{'OK ' if found else 'FAIL'}] {label}: {path}")
    if not found:
        print(f'         stderr: {r.stderr.strip()[:200]}')
print()
if check_c_ok:
    print('  CHECK C PASS -- both aux tables resolve + list on this CDR. No code edit needed.')
else:
    print('  !! CHECK C FAIL -- a derived aux path did not list. This is NOT a code-migration')
    print('     issue (the driver auto-derives). Likely causes:')
    print('       (a) env not CDR-wired -> use a Standard Analysis env (not featherweight),')
    print('           and confirm no "cdrv8 - R9 not found" on startup; re-run this cell.')
    print('       (b) AoU reorganized the v9 aux layout under /wgs/short_read/snpindel/ ->')
    print('           escalate; do NOT fire compute.')



In [ ]:
# Cell 3 — Catastrophe MT inventory
MTS = [
    ('mt_afr_qc.mt', 'AFR primary (D-M3-07)'),
    ('mt_afr_pca_selfid_qc.mt', 'AFR sensitivity'),
    ('mt_eur_qc.mt', 'EUR parity'),
]
print('=== MT inventory ===')
for mt, label in MTS:
    uri = f'gs://{BUCKET}/ld/{mt}'
    r = subprocess.run(['gsutil', 'ls', uri], capture_output=True, text=True)
    present = r.returncode == 0
    print(f'  {mt:32s} ({label}): {"PRESENT" if present else "ABSENT"}')
    if present:
        du = subprocess.run(['gsutil', 'du', '-s', uri], capture_output=True, text=True)
        if du.returncode == 0:
            size_bytes = int(du.stdout.split()[0])
            print(f'    Total size: {size_bytes:,} bytes ({size_bytes / 10**6:.2f} MB)')
        # Check entries/entries/parts/ — the catastrophe discriminator
        ep = subprocess.run(
            ['gsutil', 'du', '-s', f'{uri}/entries/entries/parts/'],
            capture_output=True, text=True,
        )
        if ep.returncode == 0:
            ep_bytes = int(ep.stdout.split()[0])
            status = 'POPULATED' if ep_bytes > 10**9 else f'TOO SMALL ({ep_bytes:,} bytes)'
            print(f'    entries/entries/parts/: {status}')
        else:
            print(f'    entries/entries/parts/: ABSENT — catastrophe pattern signature')



In [ ]:
# Cell 4 — Hypothesis distinguisher: _SUCCESS mtime test
# Per [[feedback_w1_catastrophe_hypothesis_distinguisher]]:
#
#   If mtime BEFORE 2026-05-20 22:30:00 UTC (the workbench Pause kill time)
#     -> debug-doc Hail-finalize-on-empty-contents hypothesis HOLDS
#
#   If mtime AT OR AFTER 2026-05-20 22:30:00 UTC
#     -> Carter's kill-interrupted-writes hypothesis HOLDS
#
# MT #3 (mt_eur_qc.mt) absence is uncontested under either hypothesis.
import datetime
import re

KILL_UTC = datetime.datetime(2026, 5, 20, 22, 30, 0, tzinfo=datetime.timezone.utc)
STAGE_36_UTC = datetime.datetime(2026, 5, 19, 14, 30, 0, tzinfo=datetime.timezone.utc)
STAGE_45_UTC = datetime.datetime(2026, 5, 20, 0, 0, 0, tzinfo=datetime.timezone.utc)

print('=== _SUCCESS mtime hypothesis test ===')
print(f'  Kill (workbench Pause): {KILL_UTC.isoformat()}')
print(f'  Stage 36 expected end:  {STAGE_36_UTC.isoformat()} (MT #1 if debug-doc theory)')
print(f'  Stage 45 expected end:  ~{STAGE_45_UTC.isoformat()} (MT #2 if debug-doc theory)')
print()
for mt, label in MTS:
    uri = f'gs://{BUCKET}/ld/{mt}/_SUCCESS'
    r = subprocess.run(['gsutil', 'ls', '-l', uri], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  {mt}: no _SUCCESS marker (write never finalized)')
        continue
    # gsutil ls -l output format: '   SIZE  TIMESTAMP  URI'
    m = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z?)', r.stdout)
    if not m:
        print(f'  {mt}: could not parse mtime from gsutil output: {r.stdout!r}')
        continue
    mtime_str = m.group(1).rstrip('Z')
    mtime = datetime.datetime.fromisoformat(mtime_str).replace(tzinfo=datetime.timezone.utc)
    print(f'  {mt}: _SUCCESS mtime = {mtime.isoformat()}')
    if mtime < KILL_UTC:
        print(f'    -> BEFORE kill: debug-doc Hail-finalize hypothesis HOLDS for this MT')
    else:
        print(f'    -> AT/AFTER kill: Carter kill-as-culprit hypothesis HOLDS for this MT')

print()
print('=== Hypothesis verdict ===')
print('See output above. Cross-reference with Abby Doyle reply on Zendesk #57144 (if available)')
print('to triangulate the diagnosis with AoU engineering team findings.')


## Action routing

Based on Cells 1-4 output, route to the next action:

| Cell 1 finding | Cell 3 finding | Cell 4 finding | Next action |
|---|---|---|---|
| Track 4 patches present + clone caught up | Catastrophe MTs still empty | _SUCCESS mtimes before kill | Debug-doc Hail-finalize verified; brief Abby's team if not already aware; chr22 smoke fire on RW 2.0 to validate Track 4 assertions under live Hail |
| Track 4 patches present + clone caught up | Catastrophe MTs still empty | _SUCCESS mtimes at/after kill | Carter's kill-as-culprit verified; re-fire strategy = let it complete without interruption; chr22 smoke as safety net |
| Track 4 patches present + clone caught up | Catastrophe MTs ABSENT (bucket purged) | n/a | Forensic evidence gone; restore from NCSU mirror at `.planning/quick/260521-w1-catastrophe-handoff/forensic-mirror/`; notify Abby; proceed to chr22 smoke or 1000G pivot |
| Track 4 patches present + clone caught up | Catastrophe MTs POPULATED | n/a | Surprising — bucket has been re-populated since 2026-05-21 inspection; halt and investigate before any compute fires |
| Track 4 patches missing | n/a | n/a | Clone stale or wrong branch; `git pull origin m3-W2-aou-deltas`; re-fire AOU-0 from Cell 1 |
| Env vars missing | n/a | n/a | RW platform issue; halt; investigate via AoU support |

Whatever Cell 4 reports, the Track 4 patches landed 2026-05-28 (commits `59e914b..bfe5f0e` on `origin/m3-W2-aou-deltas`) DEFEND AGAINST BOTH hypotheses — every future `mt.checkpoint()` call self-validates `count_rows() + count_cols() > 0` and the auto-resume gate checks `entries/entries/parts/` size, not just `_SUCCESS`.

**Once routing decision is made**, write a brief summary to `.planning/quick/260528-l8r-stage-aou-pre-check-chr22-smoke-aou-2-4-/post-precheck-routing-decision.txt` and commit before any compute fires.
